## Setup
Required packages: numpy, os, tifffile, stardist, matplotlib. 
Stardist also has some strange dependecy requirements; highly recommend setting up new conda environment (conda create env -n stardist) before pip installing.

Goals: Use manually segmented masks to optimize parameters, then run on whole dataset.

In [ ]:
#if set up env called stardist, can activate here
%conda init
%conda activate stardist_env

In [ ]:
#install all dependencies - RUN THIS CELL ONCE
#if doesn't work, try installing in console; just remove %

#stardist has dependency on tensorflow, make sure to use python 3.11 
# and install tensorflow first

%pip install tensorflow
%pip install stardist

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [75]:
#imports 
import re
import os
import numpy as np
import pandas as pd
import glob
from stardist.models import StarDist2D
from tifffile import imwrite, imread
from pathlib import Path
from skimage.measure import label
from stardist.matching import matching_dataset
from csbdeep.utils import normalize

## Ground truth predictions

Using manually segmented masks, either through cellpose GUI (recommended for ease of access) OR ImageJ/napari/other image processing software. Change to a format compatible with stardist. Then, predict using model and optimize.

In [28]:
#import directories
anno_dir = Path("manual_masks") #manually annotated masks (from cellpose)
mask_dir = Path("manual_masks/stardist_masks") #annotated masks for stardist
raw_dir = "fluorescence_data" #raw data directory
save_dir = "stardist_masks" #save directory

In [ ]:
##Changes cellpose-labeled manual masks to stardist-compatible masks. This only needs to be run once.

#find manually segmented files in anno_dir
seg_files = list(anno_dir.glob('*_seg.npy'))

for f in seg_files:
    #load directory per cellpose documentation
    anno = np.load(f, allow_pickle=True).item()
    masks = anno['masks'].astype(np.uint16)

    #define and save new filename
    new_filename = f.name.replace('_seg.npy', '_star.tif') #seg = segment, cellpose; star = stardist
    save_path = mask_dir / new_filename
    
    # write to tiff file
    imwrite(str(save_path), masks)
    print(f"converted {f.name} to {new_filename}")

converted Phase_Hn249_00001_seg.npy to Phase_Hn249_00001_star.tif
converted Phase_Hn249_00002_seg.npy to Phase_Hn249_00002_star.tif
converted Phase_Hn249_00010_seg.npy to Phase_Hn249_00010_star.tif
converted Phase_Hn249_00011_seg.npy to Phase_Hn249_00011_star.tif
converted Phase_Hn249_00012_seg.npy to Phase_Hn249_00012_star.tif
converted Phase_Hn249_00016_seg.npy to Phase_Hn249_00016_star.tif
converted Phase_Hn249_00025_seg.npy to Phase_Hn249_00025_star.tif
converted Phase_Hn249_00030_seg.npy to Phase_Hn249_00030_star.tif


In [30]:
# List files in the directory to verify what is actually there
print(os.listdir(raw_dir))
print(os.listdir(mask_dir))

# Explicit list of indices to process (i.e. which ones were manually annotated)
target_indices = ['00001', '00002', '00010', 
                  '00011', '00012', '00016', 
                  '00025', '00030']

['Image_Hn249_150ms_(00001).tif', 'Image_Hn249_150ms_(00002).tif', 'Image_Hn249_150ms_(00003).tif', 'Image_Hn249_150ms_(00004).tif', 'Image_Hn249_150ms_(00005).tif', 'Image_Hn249_150ms_(00006).tif', 'Image_Hn249_150ms_(00007).tif', 'Image_Hn249_150ms_(00008).tif', 'Image_Hn249_150ms_(00009).tif', 'Image_Hn249_150ms_(00010).tif', 'Image_Hn249_150ms_(00011).tif', 'Image_Hn249_150ms_(00012).tif', 'Image_Hn249_150ms_(00013).tif', 'Image_Hn249_150ms_(00014).tif', 'Image_Hn249_150ms_(00015).tif', 'Image_Hn249_150ms_(00016).tif', 'Image_Hn249_150ms_(00017).tif', 'Image_Hn249_150ms_(00018).tif', 'Image_Hn249_150ms_(00019).tif', 'Image_Hn249_150ms_(00020).tif', 'Image_Hn249_150ms_(00021).tif', 'Image_Hn249_150ms_(00022).tif', 'Image_Hn249_150ms_(00023).tif', 'Image_Hn249_150ms_(00024).tif', 'Image_Hn249_150ms_(00025).tif', 'Image_Hn249_150ms_(00026).tif', 'Image_Hn249_150ms_(00027).tif', 'Image_Hn249_150ms_(00028).tif', 'Image_Hn249_150ms_(00029).tif', 'Image_Hn249_150ms_(00030).tif', 'Image_Hn

In [76]:
model = StarDist2D.from_pretrained('2D_versatile_fluo')

Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [77]:
# setting up empty lists + IoU thresholds
all_gt_masks = []
all_pred_masks = []
all_ap_scores = [] 
thresholds = [0.5, 0.75, 0.9]
csv_records_star = []

for idx in target_indices:
    # filenames, filepaths
    raw_name = f"Image_Hn249_150ms_({idx}).tif"
    mask_name = f"Phase_Hn249_{idx}_star.tif"
    raw_path = os.path.join(raw_dir, raw_name)
    mask_path = os.path.join(mask_dir, mask_name)
    # err checking
    if not os.path.exists(raw_path) or not os.path.exists(mask_path):
        print(f"Skipping ID {idx}: File not found.")
        continue
    # load images and ground truth. Normalize image so intensity ranges from 0 - 1 (required for stardist)
    img = imread(raw_path)
    gt_mask = label(imread(mask_path)) 
    img_norm = normalize(img, 1, 99.8)
    
    # RUN MODEL - change parameters as necessary
    labels, _ = model.predict_instances(img_norm, 
                                        scale = 1.2,
                                        prob_thresh=0.5, 
                                        nms_thresh=0.05,  
                                        n_tiles=(2, 2))
    
    # compute metrics of interest
    image_ap = []
    stats_at_50 = matching_dataset([gt_mask], [labels], thresh=0.5, show_progress=False)
    for t in thresholds:
        stat = matching_dataset([gt_mask], [labels], thresh=t, show_progress=False)
        image_ap.append(stat.accuracy)

    # append data to list to later be stored as csv
    csv_records_star.append({
        'id': idx,
        'gt_count': len(np.unique(gt_mask)) - 1, 
        'pred_count': len(np.unique(labels)) - 1,
        'ap50': image_ap[0],
        'ap75': image_ap[1],
        'ap90': image_ap[2],
        'tp50': stats_at_50.tp,
        'fp50': stats_at_50.fp,
        'fn50': stats_at_50.fn
    })

    # append to lists to quickly compute average metrics in next cell
    all_ap_scores.append(image_ap)
    all_gt_masks.append(gt_mask)
    all_pred_masks.append(labels)
    
    # save indiviudal masks
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    imwrite(os.path.join(save_dir, f"pred_{idx}.tif"), labels.astype(np.uint16))
    
    print(f"AP50: {image_ap[0]:.3f} \
          # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1")

#save data to csv
df_stardist = pd.DataFrame(csv_records_star)
df_stardist.to_csv('stardist_metrics_detailed.csv', index=False)

100%|██████████| 4/4 [00:00<00:00,  5.65it/s]


AP50: 1.000           # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1


100%|██████████| 4/4 [00:00<00:00,  6.07it/s]


AP50: 0.750           # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1


100%|██████████| 4/4 [00:00<00:00,  5.96it/s]


...

AP50: 0.083           # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1


100%|██████████| 4/4 [00:00<00:00,  6.24it/s]


AP50: 0.200           # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1


100%|██████████| 4/4 [00:00<00:00,  6.12it/s]


AP50: 0.143           # of cells: len(np.unique(gt_mask)) - 1/len(np.unique(labels)) - 1


## Metrics of ground truth predictions

To evaluate the efficacy of the model, we use a mAP at several IoU thresholds (0.5, 0.75 and 0.85). 

In [ ]:
#aggregate metrics for all datasets
#create np array w/ all scores
ap_array = np.array(all_ap_scores)
mean_ap = ap_array.mean(axis=0)

print("AGGREGATE PERFORMANCE METRICS")
#print the mAP at different IoU thresholds
for i, t in enumerate(thresholds):
    print(f"mAP @ IoU {t}: {mean_ap[i]:.3f}")

# final data analysis summary
gt_counts = [len(np.unique(m)) - 1 for m in all_gt_masks]
pred_counts = [len(np.unique(m)) - 1 for m in all_pred_masks]

print("DATA ANALYSIS SUMMARY")
print(f"total # of cells (GT): {sum(gt_counts)}")
print(f"# of cells from model: {sum(pred_counts)}")

# average error calculation
avg_error = np.mean([abs(g - p) for g, p in zip(gt_counts, pred_counts)])
print(f"error: {avg_error:.1f} cells")

AGGREGATE PERFORMANCE METRICS
mAP @ IoU 0.5: 0.344
mAP @ IoU 0.75: 0.062
mAP @ IoU 0.9: 0.000
DATA ANALYSIS SUMMARY
total # of cells (GT) 34
# of cells from model 31
error: 0.9 cells


## Run model on all data

In [83]:
#initialize model again, in case not done earlier
model = StarDist2D.from_pretrained('2D_versatile_fluo')

Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [86]:
all_files = sorted(glob.glob(os.path.join(raw_dir, "*.tif")))

for raw_path in all_files:
    #open filepath, read image and normalize 
    fname = os.path.basename(raw_path)
    img = imread(raw_path)
    img_norm = normalize(img, 1, 99.8)
    
    #run model
    labels, _ = model.predict_instances(img_norm, 
                                        scale = 1.2,
                                        prob_thresh=0.5, 
                                        nms_thresh=0.05,  
                                        n_tiles=(2, 2))
    
    # save mask
    save_path = os.path.join(save_dir, f"mask_{fname}.tif")
    imwrite(save_path, labels.astype(np.uint16))
    
    # print cell count
    n_cells = len(np.unique(labels)) - 1
    print(f"detected: {n_cells} cells")

100%|██████████| 4/4 [00:00<00:00,  5.93it/s]


detected: 3 cells


100%|██████████| 4/4 [00:00<00:00,  5.84it/s]


detected: 3 cells


100%|██████████| 4/4 [00:00<00:00,  5.83it/s]


...

detected: 2 cells


100%|██████████| 4/4 [00:00<00:00,  6.50it/s]


detected: 3 cells


100%|██████████| 4/4 [00:00<00:00,  6.46it/s]

detected: 2 cells
